# Short Transformer GPU Training on Colab (`0.2.1`)

This notebook runs a short end-to-end transformer training on a GPU-enabled Colab runtime.

It uses the repository test data and a small Hugging Face backbone so the run stays lightweight, while training a bit longer than the minimal smoke test.

It runs:
- repository clone
- dependency install
- CUDA runtime check
- one short GPU training run with a small train/validation split
- prediction with the freshly trained model
- quick inspection of saved model artifacts


In [ ]:
%cd /content
!rm -rf /content/job-ads-classifier /content/models
!git clone --branch main https://github.com/OJALAB/job-ads-classifier.git /content/job-ads-classifier
%cd /content/job-ads-classifier
!python -m pip install --upgrade pip
!pip install -r requirements-colab-0.2.0.txt

In [ ]:
import subprocess

import torch

subprocess.run(["nvidia-smi"], check=False)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device_name:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), (
    "This notebook expects a GPU runtime. In Colab choose Runtime > Change runtime type > GPU."
)


## Training configuration

The next cell uses the tiny test dataset from `tests/data` and a compact transformer model for a short but slightly more realistic GPU training run.

The notebook uses the Python API for fitting so it can pass an explicit validation split.


In [ ]:
from pathlib import Path
import subprocess

def run_command(command):
    print("$", " ".join(command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

# Optional Google Drive mount if you want to persist the trained model.
# from google.colab import drive
# drive.mount("/content/drive")

MODEL_DIR = "/content/models/transformer-demo-gpu"
PRED_PATH = "/content/predictions-trained-gpu.txt"
TRANSFORMER_MODEL = "google/bert_uncased_L-2_H-128_A-2"
MAX_EPOCHS = "3"
BATCH_SIZE = "2"
MAX_SEQUENCE_LENGTH = "32"
PRECISION = "16-mixed"
THREADS = "1"

Path("/content/models").mkdir(parents=True, exist_ok=True)
print("MODEL_DIR =", MODEL_DIR)
print("PRED_PATH =", PRED_PATH)
print("TRANSFORMER_MODEL =", TRANSFORMER_MODEL)


In [ ]:
from job_offers_classifier.job_offers_classfier import TransformerJobOffersClassifier
from job_offers_classifier.job_offers_utils import create_hierarchy
from job_offers_classifier.load_save import load_texts, load_to_df

hierarchy = create_hierarchy(load_to_df("tests/data/classes.tsv"))
x_all = load_texts("tests/data/x_train.txt")
y_all = load_texts("tests/data/y_train.txt")

x_train = x_all[:6]
y_train = y_all[:6]
x_val = x_all[6:]
y_val = y_all[6:]

assert x_train and y_train
assert x_val and y_val

model = TransformerJobOffersClassifier(
    model_dir=MODEL_DIR,
    hierarchy=hierarchy,
    transformer_model=TRANSFORMER_MODEL,
    modeling_mode="bottom-up",
    learning_rate=5e-5,
    max_epochs=int(MAX_EPOCHS),
    batch_size=int(BATCH_SIZE),
    max_sequence_length=int(MAX_SEQUENCE_LENGTH),
    devices=1,
    accelerator="gpu",
    precision=PRECISION,
    threads=int(THREADS),
    verbose=True,
)

model.fit(y_train, x_train, y_val=y_val, X_val=x_val)


In [ ]:
for path in sorted(Path(MODEL_DIR).rglob("*")):
    if path.is_file():
        print(path.relative_to(MODEL_DIR))


In [ ]:
run_command([
    "python",
    "main.py",
    "predict",
    "TransformerJobOffersClassifier",
    "-x",
    "tests/data/x_test.txt",
    "-m",
    MODEL_DIR,
    "-p",
    PRED_PATH,
    "-A",
    "gpu",
    "-P",
    PRECISION,
    "-T",
    THREADS,
])


In [ ]:
run_command(["head", "-n", "5", PRED_PATH])
run_command(["head", "-n", "5", f"{PRED_PATH}.map"])
